# GIN — How Powerful are Graph Neural Networks? (2018)

_Papers · Graph Neural Networks_

**A graph network is exactly as discriminative as its neighbour-aggregator is injective; SUM-over-neighbours fed through an MLP is injective, making GIN as powerful as the Weisfeiler-Lehman graph-isomorphism test.**

---

This notebook is a practice scaffold for the **GIN — How Powerful are Graph Neural Networks? (2018)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- 0. Worked example: SUM vs MEAN vs MAX on two different neighbour multisets. ---
# Colours as one-hot:  red=[1,0], blue=[0,1].
red, blue = torch.tensor([1.,0.]), torch.tensor([0.,1.])
X  = torch.stack([red, red, blue])    # multiset {red, red, blue}  (node v)
Xp = torch.stack([red, blue])         # multiset {red, blue}       (node v')
for name, agg in [("SUM",  lambda T: T.sum(0)),
                  ("MEAN", lambda T: T.mean(0)),
                  ("MAX",  lambda T: T.max(0).values)]:
    a, b = agg(X), agg(Xp)
    print(f"{name:4s}  {{r,r,b}}={a.tolist()}  {{r,b}}={b.tolist()}  "
          f"-> {'DISTINGUISHED' if not torch.allclose(a,b) else 'CONFUSED'}")
# Harder pair where MEAN also fails: {r,r} vs {r}
X2, X2p = torch.stack([red, red]), red.unsqueeze(0)
for name, agg in [("SUM",  lambda T: T.sum(0)),
                  ("MEAN", lambda T: T.mean(0)),
                  ("MAX",  lambda T: T.max(0).values)]:
    a, b = agg(X2), agg(X2p)
    print(f"{name:4s}  {{r,r}}={a.tolist()}  {{r}}={b.tolist()}  "
          f"-> {'DISTINGUISHED' if not torch.allclose(a,b) else 'CONFUSED'}")
# SUM distinguishes both pairs; MEAN/MAX confuse {r,r} vs {r}; MAX also confuses {r,r,b} vs {r,b}.
print()

# --- 1. A tiny graph-classification task: label = 1 iff some node has >= 3 red neighbours. ---
# This is a COUNT rule -> needs an injective (SUM) aggregator; MEAN cannot recover the count.
def make_graph(n, p_edge, n_red, seed):
    g = torch.Generator().manual_seed(seed)
    A = (torch.rand(n, n, generator=g) < p_edge).float()
    A = torch.triu(A, 1); A = A + A.t()                 # symmetric, no self-loops in A
    feats = torch.zeros(n, 2); feats[:, 1] = 1.0         # all blue [0,1]
    red_idx = torch.randperm(n, generator=g)[:n_red]
    feats[red_idx] = torch.tensor([1., 0.])              # paint some nodes red [1,0]
    red_neigh = (A @ feats[:, 0:1]).squeeze(1)           # # red neighbours per node
    label = int((red_neigh >= 3).any().item())           # class 1 iff any node has >=3 red neighbours
    return A, feats, label

def make_dataset(num, seed0=0):
    data = []
    for s in range(num):
        n = int(torch.randint(6, 11, (1,)).item())
        A, X, y = make_graph(n, p_edge=0.45, n_red=int(torch.randint(2, 6, (1,)).item()), seed=seed0 + s)
        data.append((A, X, y))
    return data

torch.manual_seed(1)
train = make_dataset(160, seed0=0)
test  = make_dataset(60,  seed0=1000)

# --- 2. The GIN layer, by hand:  MLP( (1+eps)*H + A @ H ).  A @ H is the neighbour SUM. ---
class GINLayer(nn.Module):
    def __init__(self, c_in, c_out, eps=0.0):
        super().__init__()
        self.eps = nn.Parameter(torch.tensor(float(eps)))     # learnable (1+eps) self-weight
        self.mlp = nn.Sequential(nn.Linear(c_in, c_out), nn.ReLU(), nn.Linear(c_out, c_out))
    def forward(self, A, H):
        neigh = A @ H                                         # SUM_{u in N(v)} h_u   (Eq. 4.1)
        return self.mlp((1 + self.eps) * H + neigh)           # MLP( (1+eps)h_v + sum neighbours )

class GIN(nn.Module):
    def __init__(self, c_in, c_hid, n_classes, K=2):
        super().__init__()
        dims = [c_in] + [c_hid] * K
        self.layers = nn.ModuleList(GINLayer(dims[i], dims[i+1]) for i in range(K))
        # READOUT (Eq. 4.2): sum-pool each layer's nodes, concat across layers (incl. input).
        self.head = nn.Linear(c_in + c_hid * K, n_classes)
    def forward(self, A, X):
        H = X; pooled = [X.sum(0)]                            # layer-0 graph pooling
        for layer in self.layers:
            H = torch.relu(layer(A, H)); pooled.append(H.sum(0))
        return self.head(torch.cat(pooled))                  # graph logits

def run(dataset_train, dataset_test, epochs=40):
    torch.manual_seed(0)
    model = GIN(c_in=2, c_hid=16, n_classes=2, K=2)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(epochs):
        model.train()
        for A, X, y in dataset_train:
            opt.zero_grad()
            logit = model(A, X).unsqueeze(0)
            F.cross_entropy(logit, torch.tensor([y])).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = sum(int(model(A, X).argmax().item() == y) for A, X, y in dataset_test) / len(dataset_test)
    return acc

acc = run(train, test)
print(f"GIN (SUM aggregator) test accuracy on the count rule = {acc:.3f}")
# GIN with SUM solves the >=3-red-neighbours rule (~1.0); a MEAN variant cannot (see CODEVIZ).
# (our small synthetic run -- not the paper's reported MUTAG/PROTEINS/REDDIT numbers)

## Visualize the data & results

_On a graph-classification task whose label is a COUNT rule ('some node has &ge; 3 red neighbours'), does the injective SUM aggregator solve it &mdash; and what happens when we ablate SUM &rarr; MEAN (divide by degree, the GCN-style average)?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np

# Does the injective SUM aggregator beat MEAN on a COUNT-based graph label?
# Label = 1 iff some node has >= 3 red neighbours. SUM keeps the count; MEAN divides it away.
def make_graph(n, p_edge, n_red, seed):
    g = torch.Generator().manual_seed(seed)
    A = (torch.rand(n, n, generator=g) < p_edge).float()
    A = torch.triu(A, 1); A = A + A.t()
    feats = torch.zeros(n, 2); feats[:, 1] = 1.0
    feats[torch.randperm(n, generator=g)[:n_red]] = torch.tensor([1., 0.])
    label = int(((A @ feats[:, 0:1]).squeeze(1) >= 3).any().item())
    return A, feats, label

def make_dataset(num, seed0):
    out = []
    for s in range(num):
        n = int(torch.randint(6, 11, (1,)).item())
        out.append(make_graph(n, 0.45, int(torch.randint(2, 6, (1,)).item()), seed0 + s))
    return out

torch.manual_seed(1)
train = make_dataset(160, 0); test = make_dataset(60, 1000)

class GIN(nn.Module):
    def __init__(s, mode):           # mode = "sum" (GIN) or "mean" (ablation)
        super().__init__(); s.mode = mode; K = 2; c_in, c_hid = 2, 16
        s.eps = nn.ParameterList([nn.Parameter(torch.tensor(0.0)) for _ in range(K)])
        dims = [c_in, c_hid, c_hid]
        s.mlps = nn.ModuleList(nn.Sequential(nn.Linear(dims[i], dims[i+1]), nn.ReLU(),
                                             nn.Linear(dims[i+1], dims[i+1])) for i in range(K))
        s.head = nn.Linear(c_in + c_hid * K, 2)
    def agg(s, A, H):
        if s.mode == "sum":  return A @ H
        deg = A.sum(1, keepdim=True).clamp(min=1)
        return (A @ H) / deg                                  # ablation: neighbour MEAN
    def forward(s, A, X):
        H = X; pooled = [X.sum(0)]
        for eps, mlp in zip(s.eps, s.mlps):
            H = torch.relu(mlp((1 + eps) * H + s.agg(A, H))); pooled.append(H.sum(0))
        return s.head(torch.cat(pooled))

def run(mode, epochs=40):
    torch.manual_seed(0); m = GIN(mode); opt = torch.optim.Adam(m.parameters(), lr=0.01); accs = []
    for _ in range(epochs):
        m.train()
        for A, X, y in train:
            opt.zero_grad(); F.cross_entropy(m(A, X).unsqueeze(0), torch.tensor([y])).backward(); opt.step()
        m.eval()
        with torch.no_grad():
            accs.append(sum(int(m(A, X).argmax().item() == y) for A, X, y in test) / len(test))
    return accs

s_acc = run("sum"); m_acc = run("mean")
idx = np.linspace(0, 39, 20).astype(int)
print("SUM  (GIN):", [[int(i), round(s_acc[i], 3)] for i in idx])
print("MEAN (abl):", [[int(i), round(m_acc[i], 3)] for i in idx])
# SUM -> ~100%; MEAN stuck near chance (~55%): it cannot recover the neighbour count.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a GIN-style network that classifies tiny graphs by a structural rule:
            "graph is class 1 if it contains a node with &ge; 3 neighbours of colour red, else class 0." Train it
            once with the neighbour SUM aggregator and once with the identical network but a neighbour
            MEAN aggregator (sum divided by degree). What happens to each, and what does the contrast prove?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap exactly one thing: the aggregator. Keep the same MLP sizes, self-term, READOUT, optimizer, labels, and epochs; change only A @ H (sum) to a degree-normalized A @ H (mean). — _An honest ablation changes one factor, so any performance gap is attributable to sum-vs-mean injectivity, not to capacity or tuning._
- Note the task is a count rule ("&ge; 3 red neighbours"). The SUM net's neighbour summary contains the raw count of red, so a downstream unit can threshold it; accuracy approaches 100%. — _SUM is injective on multisets, so the count of red neighbours survives into the node vector._
- The MEAN net divides by degree, so a node with 3 red out of 6 neighbours looks identical to one with 1 red out of 2 (both 0.5 red). It cannot recover the count and stalls near chance on this rule. — _MEAN keeps only the proportion of red, not the absolute count, so the count-based label is unrecoverable &mdash; the exact non-injectivity the paper proves._

**Answer:** The SUM network learns the count rule and reaches near-perfect accuracy; the MEAN
                 network is stuck near chance because dividing by degree erases the absolute number of red
                 neighbours ($\{r,r,r\}$ out of 6 looks like $\{r\}$ out of 2). Since the only change was
                 sum vs mean, this isolates injective (SUM) aggregation &mdash; not extra parameters &mdash;
                 as the source of the extra expressive power, exactly the Theorem 3 / &sect;5 story. The CODEVIZ
                 panel shows this gap.

</details>

**Problem 2.** Two nodes have neighbour multisets $X = \{\text{blue},\text{blue},\text{green}\}$ and
            $X' = \{\text{blue},\text{green},\text{green}\}$. Using one-hot $\text{blue}=[1,0]$,
            $\text{green}=[0,1]$, compute the SUM, MEAN, and MAX summaries of each and say which aggregators
            distinguish $X$ from $X'$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- SUM: $X\to[1,0]+[1,0]+[0,1]=[2,1]$; $X'\to[1,0]+[0,1]+[0,1]=[1,2]$. — _Summing keeps the full counts, so $[2,1]\ne[1,2]$ &mdash; distinguished._
- MEAN: $X\to[2,1]/3=[0.667,0.333]$; $X'\to[1,2]/3=[0.333,0.667]$. — _Here the proportions differ (2:1 vs 1:2), so mean happens to distinguish them &mdash; but it would fail on $\{b,b\}$ vs $\{b\}$ (both $[1,0]$)._
- MAX: $X\to[\max(1,1,0),\max(0,0,1)]=[1,1]$; $X'\to[1,1]$. — _Max records only presence (both colours appear in both), not counts, so $[1,1]=[1,1]$ &mdash; confused._

**Answer:** SUM gives $[2,1]$ vs $[1,2]$ (distinguished); MEAN gives $[0.667,0.333]$ vs
                 $[0.333,0.667]$ (distinguished here, because proportions differ, but mean is non-injective in
                 general); MAX gives $[1,1]$ vs $[1,1]$ (confused &mdash; it sees only that both colours are
                 present). This matches Figure 2's ranking: SUM keeps the full multiset, MEAN keeps only the
                 distribution, MAX keeps only the set.

</details>

**Problem 3.** A colleague replaces GIN's two-layer MLP with a single nn.Linear "to simplify," keeping the
            SUM aggregator and the $(1+\epsilon)$ self-term. Their model now fails to separate some graphs that
            full GIN separates. Why &mdash; isn't the SUM the important part?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall Theorem 3 needs the whole aggregate-and-combine map to be injective, and Lemma 5 gives an $f$ making $\sum f(x)$ injective &mdash; but $f$ must be learnable as a sufficiently rich function. — _The SUM is injective only after the right per-element map $f$ is applied; a poor $f$ can re-introduce collisions._
- Apply Lemma 7: there exist distinct multisets $X_1\ne X_2$ with $\sum_{x\in X_1}\mathrm{ReLU}(Wx)=\sum_{x\in X_2}\mathrm{ReLU}(Wx)$ for every linear $W$. — _A single linear (or single-perceptron) layer cannot realize the injective $f$; some multisets stay merged no matter the weights._
- Restore the MLP (Linear &rarr; ReLU &rarr; Linear), which by universal approximation can model the needed $f$. — _The hidden nonlinearity between two linears is exactly what lets the network learn an injective per-element map._

**Answer:** Because the SUM is injective only once a rich enough per-element map $f$ is applied first,
                 and Lemma 7 proves a single linear layer cannot be that map: there are distinct multisets it merges
                 for every weight matrix. GIN therefore needs a real MLP (at least two linears with a
                 nonlinearity between) so universal approximation can realize the injective $f$. SUM is necessary but
                 not sufficient &mdash; SUM and MLP together give Theorem 3's guarantee.

</details>